In [1]:
import pandas as pd

df = pd.read_csv("raw_inventory.csv")
df.head()

,Month,Product_ID,Product_Name,Plant,Opening_Stock,Receipts,Closing_Stock
0,2026-01-01,P001,Resin A,Plant 1,555,180,514
1,2026-02-01,P001,Resin A,Plant 1,851,825,882
2,2026-03-01,P001,Resin A,Plant 1,1411,28,679
3,2026-04-01,P001,Resin A,Plant 1,1362,577,1808
4,2026-05-01,P001,Resin A,Plant 1,585,856,608


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Month          90 non-null     str  
 1   Product_ID     90 non-null     str  
 2   Product_Name   90 non-null     str  
 3   Plant          90 non-null     str  
 4   Opening_Stock  90 non-null     int64
 5   Receipts       90 non-null     int64
 6   Closing_Stock  90 non-null     int64
dtypes: int64(3), str(4)
memory usage: 5.1 KB


In [3]:
df.describe()

,Opening_Stock,Receipts,Closing_Stock
count,90.000000,90.000000,90.000000
mean,935.266667,431.233333,915.366667
std,453.392525,256.023614,524.103525
min,252.000000,0.000000,0.000000
25%,552.750000,210.250000,519.000000
50%,924.500000,444.500000,921.500000
75%,1361.500000,623.000000,1293.000000
max,1750.000000,894.000000,2049.000000


In [4]:
df.isnull().sum()

Month            0
Product_ID       0
Product_Name     0
Plant            0
Opening_Stock    0
Receipts         0
Closing_Stock    0
dtype: int64

In [5]:
# Convert date column
df['Month'] = pd.to_datetime(df['Month'], errors='coerce')

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate keys:", df.duplicated(['Month', 'Product_ID', 'Plant']).sum())

# Check negative quantities
print("Negative opening stock:", (df['Opening_Stock'] < 0).sum())
print("Negative receipts:", (df['Receipts'] < 0).sum())
print("Negative closing stock:", (df['Closing_Stock'] < 0).sum())

# Check closing stock greater than opening stock + receipts
print("Closing > Opening + Receipts:",
      (df['Closing_Stock'] > df['Opening_Stock'] + df['Receipts']).sum())

# Display suspicious records
print(df[df['Closing_Stock'] > df['Opening_Stock'] + df['Receipts']])

Duplicate rows: 0
Duplicate keys: 0
Negative opening stock: 0
Negative receipts: 0
Negative closing stock: 0
Closing > Opening + Receipts: 1
        Month Product_ID Product_Name    Plant  Opening_Stock  Receipts  \
76 2026-05-01       P009    Solvent G  Plant 3           1138       894   

    Closing_Stock  
76           2049  


In [6]:
# Quantity consumed during the month
df['Consumed_Qty'] = (
    df['Opening_Stock'] + df['Receipts'] - df['Closing_Stock']
)

# Change in stock over the month
df['Net_Change'] = (
    df['Closing_Stock'] - df['Opening_Stock']
)

# Stock turnover
df['Turnover'] = (
    df['Consumed_Qty'] / ((df['Opening_Stock'] + df['Closing_Stock']) / 2)
)

# Stock status
df['Stock_Status'] = pd.cut(
    df['Closing_Stock'],
    bins=[-1, 50, 1499, 10**9],
    labels=['Stockout Risk', 'Normal', 'Excess']
)

df.head()


#Consumed quantity: Opening stock plus receipts minus closing stock; negative means an impossible stock flow.



#Net change: Positive means stock increased; negative means stock decreased.



#Turnover: How fast stock is used compared to the average stock held.



#Stock status: Stockout Risk (closing stock <= 50), Normal, or Excess (closing stock >= 1500).

,Month,Product_ID,Product_Name,Plant,Opening_Stock,Receipts,Closing_Stock,Consumed_Qty,Net_Change,Turnover,Stock_Status
0,2026-01-01,P001,Resin A,Plant 1,555,180,514,221,-41,0.413471,Normal
1,2026-02-01,P001,Resin A,Plant 1,851,825,882,794,31,0.916330,Normal
2,2026-03-01,P001,Resin A,Plant 1,1411,28,679,760,-732,0.727273,Normal
3,2026-04-01,P001,Resin A,Plant 1,1362,577,1808,131,446,0.082650,Excess
4,2026-05-01,P001,Resin A,Plant 1,585,856,608,833,23,1.396479,Normal


In [7]:
# Excess inventory (closing stock >= 1500)
excess = df[df['Closing_Stock'] >= 1500].copy()
excess['Month_Name'] = excess['Month'].dt.strftime('%b-%Y')

excess.sort_values(['Product_Name', 'Month'])[['Product_Name', 'Plant', 'Month_Name', 'Closing_Stock']]

,Product_Name,Plant,Month_Name,Closing_Stock
22,Additive X,Plant 2,May-2026,1838
26,Additive X,Plant 2,Sep-2026,1520
28,Additive Y,Plant 2,Feb-2026,1638
42,Coating C,Plant 3,Jul-2026,1609
51,Coating D,Plant 3,Jul-2026,1610
60,Polymer E,Plant 1,Jul-2026,1664
68,Polymer F,Plant 2,Jun-2026,1972
69,Polymer F,Plant 2,Jul-2026,1944
3,Resin A,Plant 1,Apr-2026,1808
7,Resin A,Plant 1,Aug-2026,1640


In [8]:
# Stockout risk (closing stock <= 50)
risk = df[df['Closing_Stock'] <= 50].copy()
risk['Month_Name'] = risk['Month'].dt.strftime('%b-%Y')

risk.sort_values(['Product_Name', 'Month'])[['Product_Name', 'Plant', 'Month_Name', 'Closing_Stock']]

,Product_Name,Plant,Month_Name,Closing_Stock
21,Additive X,Plant 2,Apr-2026,0
23,Additive X,Plant 2,Jun-2026,0
25,Additive X,Plant 2,Aug-2026,12
50,Coating D,Plant 3,Jun-2026,0
66,Polymer F,Plant 2,Apr-2026,0


In [9]:
# Products with repeated stockout risk
repeat = risk.groupby(['Product_Name', 'Plant']).size().reset_index(name='Risk_Months')
repeat[repeat['Risk_Months'] > 1]

,Product_Name,Plant,Risk_Months
0,Additive X,Plant 2,3


In [10]:
# Products with both excess and stockout risk
set(excess['Product_Name']) & set(risk['Product_Name'])

{'Additive X', 'Coating D', 'Polymer F'}

In [11]:
plant_analysis = df.groupby('Plant').agg(
    Total_Opening=('Opening_Stock', 'sum'),
    Total_Receipts=('Receipts', 'sum'),
    Total_Closing=('Closing_Stock', 'sum'),
    Records=('Product_ID', 'count'),
    Excess_Cases=('Closing_Stock', lambda x: (x >= 1500).sum()),
    Stockout_Cases=('Closing_Stock', lambda x: (x <= 50).sum())
).reset_index()

plant_analysis.sort_values(
    'Stockout_Cases', ascending=False
)

,Plant,Total_Opening,Total_Receipts,Total_Closing,Records,Excess_Cases,Stockout_Cases
1,Plant 2,25600,11864,24423,27,5,4
2,Plant 3,31786,15136,31282,36,4,1
0,Plant 1,26788,11811,26678,27,4,0


In [12]:
df['Month_Period'] = df['Month'].dt.to_period('M')

monthly_analysis = df.groupby('Month_Period').agg(
    Receipts=('Receipts', 'sum'),
    Closing_Stock=('Closing_Stock', 'sum')
).reset_index()

monthly_analysis

,Month_Period,Receipts,Closing_Stock
0,2026-01,5203,8772
1,2026-02,4355,9010
2,2026-03,2261,6170
3,2026-04,3813,8265
4,2026-05,6169,13102
5,2026-06,4565,8283
6,2026-07,4363,10760
7,2026-08,3934,9097
8,2026-09,4148,8924


In [13]:
df.to_csv("clean_inventory.csv", index=False)